# Homework #2 - Sistemi Informativi Evoluti e Big Data (2025-2026)
## Analisi dei dati criminali di Los Angeles - InfluxDB

Questo notebook implementa l'analisi completa del dataset [Crime Data from 2020 to Present](https://catalog.data.gov/dataset/crime-data-from-2020-to-present) utilizzando InfluxDB.

## Setup - Installazione delle librerie necessarie

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
import time
from datetime import datetime

# Librerie InfluxDB
from influxdb_client import InfluxDBClient, Point, WritePrecision
from influxdb_client.client.write_api import WriteOptions
from influxdb_client.client.warnings import MissingPivotFunction

# Librerie per Clustering 
from sklearn.cluster import MiniBatchKMeans
from sklearn.preprocessing import StandardScaler
from river import cluster

# Disattivazione warnings
warnings.filterwarnings('ignore')
warnings.simplefilter("ignore", MissingPivotFunction)

## 1. Configurazione InfluxDB e Caricamento Dati

### 1.1 Connessione al Database

Configuriamo la connessione a InfluxDB con parametri ottimizzati per il batch writing di grandi volumi di dati.

In [63]:

# Configurazione InfluxDB
url = "http://localhost:8086"
token = "W3cPRGkq7ds7JHFNtuciiVqEVPGugFye82mykiLWWee2FMxLDflsqsHk8OJaavTS9Sjh1_H31xU4yh3J5IC9RA=="
org = "unibs"
bucket = "crime_data"

client = InfluxDBClient(url=url, token=token, org=org)
write_api = client.write_api(
    write_options=WriteOptions(
        batch_size=5_000,        # numero di punti per batch
        flush_interval=10_000,  # flush automatico ogni 10 secondi
        jitter_interval=2_000,  # aggiunge un po' di random per evitare spike
        retry_interval=5_000,   # ritenta in caso di errore temporaneo
    )
)
query_api = client.query_api()

### Funzione di supporto per "svuotare" il bucket InfluxDB

In [64]:
from influxdb_client.rest import ApiException

buckets_api = client.buckets_api()
# Cerca il bucket esistente
try:
    existing_bucket = buckets_api.find_bucket_by_name(bucket)
    
    if existing_bucket:
        print(f"Trovato bucket esistente (ID: {existing_bucket.id}). Eliminazione in corso...")
        # Elimina il bucket
        buckets_api.delete_bucket(existing_bucket)
        print("Bucket eliminato.")
    else:
        print("Il bucket non esisteva.")

    time.sleep(2) # attendi

    # Ricrea il bucket
    org_api = client.organizations_api()
    org_obj = org_api.find_organizations(org=org)[0]
    
    print(f"Creazione nuovo bucket '{bucket}' nell'organizzazione '{org}'...")
    created_bucket = buckets_api.create_bucket(bucket_name=bucket, org_id=org_obj.id)
    print(f"Bucket creato con successo (Nuovo ID: {created_bucket.id})")

except ApiException as e:
    print(f"Errore durante il reset: {e}")
except Exception as e:
    print(f"Errore generico: {e}")

Trovato bucket esistente (ID: 581940b9aedc5eba). Eliminazione in corso...
Bucket eliminato.
Bucket eliminato.
Creazione nuovo bucket 'crime_data' nell'organizzazione 'unibs'...
Bucket creato con successo (Nuovo ID: e10ab828e1f1e5d2)
Creazione nuovo bucket 'crime_data' nell'organizzazione 'unibs'...
Bucket creato con successo (Nuovo ID: e10ab828e1f1e5d2)


## 1.3 Caricamento Dati e Parsing Date

In [39]:
# Caricamento Dataset
csv_path = "Crime_Data_from_2020_to_Present.csv"
df = pd.read_csv(csv_path)

# Conversione date
# DATE OCC: data ipotizzata in cui si è verificato il crimine (usata come timestamp principale)
# Date Rptd: data della segnalazione (salvata come field aggiuntivo)
df['DATE OCC'] = pd.to_datetime(df['DATE OCC'], format='%m/%d/%Y %I:%M:%S %p', errors='coerce')
df['Date Rptd'] = pd.to_datetime(df['Date Rptd'], format='%m/%d/%Y %I:%M:%S %p', errors='coerce')

### 1.3 Funzioni di Supporto

- Pulizia dati secondo le specifiche
- Feature Engineering

In [ ]:
# def get_season(date):
#     """Determina la stagione meteorologica dalla data."""
#     month = date.month
#     if month in [3, 4, 5]: return "Spring"
#     elif month in [6, 7, 8]: return "Summer"
#     elif month in [9, 10, 11]: return "Autumn"
#     else: return "Winter"

# def get_season(date):
#     # Tupla (mese, giorno)
#     md = (date.month, date.day)
#     if (3, 21) <= md < (6, 21):
#         return "Spring"
#     elif (6, 21) <= md < (9, 21):
#         return "Summer"
#     elif (9, 21) <= md < (12, 21):
#         return "Autumn"
#     else:
#         # 12, 21 - 3, 20
#         return "Winter"

def get_season(date):
    """Determina la stagione meteorologica dalla data."""
    month = date.month
    day = date.day
    
    if (month == 12 and day >= 21) or month in [1, 2] or (month == 3 and day < 21):
        return "Winter"
    elif (month == 3 and day >= 21) or month in [4, 5] or (month == 6 and day < 21):
        return "Spring"
    elif (month == 6 and day >= 21) or month in [7, 8] or (month == 9 and day < 21):
        return "Summer"
    else:
        return "Autumn"

def get_day_night(time_occ):
    """Determina se il crimine è avvenuto di giorno o di notte.
    TIME OCC è un intero nel formato HHMM (es. 1430 -> 14:30)
    """
    hour = int(time_occ / 100)
    return "Day" if 6 <= hour <= 18 else "Night"

def data_generator(df_input):
    """
    Carica i dati dei crimini in InfluxDB simulando un flusso di dati.
    """
    
    # Contatori
    cnt_coords = 0
    cnt_date = 0
    cnt_crm = 0
    cnt_written = 0

    # Data limite: Novembre 2025 (come da consegna)
    cutoff_date = pd.Timestamp('2025-11-30')
    
    print(f"Record totali da processare: {len(df_input):,}")
    
    for idx, row in df_input.iterrows():
        # === FILTRI VALORI FUORI SCALA ===
        
        # 1. Coordinate Nulle
        if row['LAT'] == 0 and row['LON'] == 0:
            cnt_coords += 1
            continue
            
        # 2. Date Future o Nulle
        if pd.isna(row['DATE OCC']) or row['DATE OCC'] > cutoff_date:
            cnt_date += 1
            continue
            
        # 3. Tipo di Crimine non valido
        if pd.isna(row['Crm Cd']):
            cnt_crm += 1
            continue

        # === FEATURE ENGINEERING ===
        season = get_season(row['DATE OCC'])
        day_night = get_day_night(row['TIME OCC'])
        
        # Combina Date Occ con offset di sicurezza
        # Con solo DATE OCC + tutti i TAG avremmo:
            # Chiavi uniche:       481,607
            # Record con collisione: 780,789
            # Record persi:        521,144 (51.97%)

        unique_ts = row['DATE OCC'] + pd.Timedelta(microseconds = idx % 1_000_000)
        
        # === PULIZIA DATI ===
        
        # Sesso vittima: M, F o X (unknown)
        if pd.notna(row['Vict Sex']) and row['Vict Sex'] in ['M', 'F']:
            vict_sex = str(row['Vict Sex'])
        else:
            vict_sex = "X"
        
        # Etnia vittima
        vict_descent = str(row['Vict Descent']) if pd.notna(row['Vict Descent']) and row['Vict Descent'] not in ['-', ''] else "X"
        
        # Gestione Arma - WeaponFlag come TAG (True/False), WeaponDetail come FIELD opzionale (dopo aver creato il point)
        if pd.notna(row['Weapon Used Cd']):
            weapon_flag = "True"
            
            # Dall'analisi dati è emerso che 'STRONG-ARM' e 'VERBAL THREAT' sono classificati come armi.
            # Se volessimo distinguere le armi vere dalla violenza fisica:
            # if "STRONG-ARM" in weapon_detail.upper() or "VERBAL THREAT" in weapon_detail.upper():
            #     weapon_flag = "False"  # Tecnicamente "Senza Arma Esterna"
            
        else:
            weapon_flag = "False"

        # === CREAZIONE POINT ===
        point = (
            Point("crime_report")
            # --- TAG (bassa cardinalità) ---
            .tag("area_name", str(row['AREA NAME']))
            .tag("area_id", str(int(row['AREA'])))
            .tag("part", str(int(row['Part 1-2']))) # lassificazione (1=grave, 2=minore)
            .tag("weapon_flag", weapon_flag)
            .tag("season", season)
            .tag("time_period", day_night)
            .tag("vict_sex", vict_sex)
            # --- FIELD (valori numerici o alta cardinalità) ---
            .field("dr_no", int(row['DR_NO']))
            .field("lat", float(row['LAT']))
            .field("lon", float(row['LON']))
            .field("district", int(row['Rpt Dist No'])) # Codice Distretto
            .field("crime_cd", int(row['Crm Cd'])) # Codice tipo crimine
            .field("crime_desc", str(row['Crm Cd Desc']))
            .field("date_rptd", str(row['Date Rptd']))
            .field("vict_descent", vict_descent)
            .time(unique_ts, WritePrecision.US)  # US = microsecondi
        )
        
        # Età vittima solo se valida (> 0)
        if pd.notna(row['Vict Age']) and row['Vict Age'] > 0:
            point.field("vict_age", int(row['Vict Age']))

        # Dettaglio arma solo se presente
        if weapon_flag == "True":
            point.field("weapon_detail", str(row['Weapon Desc']))
            
        # Scrivi punto su InfluxDB
        write_api.write(bucket=bucket, org=org, record=point)
        cnt_written += 1
        
        # Log di progresso ogni 100k record
        if cnt_written % 100000 == 0:
            print(f"  {cnt_written:,} record scritti...")

    # Flush e close per garantire completamento scritture
    write_api.flush()
    write_api.close()
    
    print(f"\nnCaricamento completato!")
    print(f"   Crimini scritti: {cnt_written:,}")
    print(f"   Record scartati:")
    print(f"     - Coordinate nulle: {cnt_coords:,}")
    print(f"     - Data non valida: {cnt_date:,}")
    print(f"     - Codice crimine mancante: {cnt_crm:,}")

### 1.4 Caricamento Dati in InfluxDB

In [66]:
# Scrittura su InfluxDB
print("Inizio caricamento dati con filtro in tempo reale...")
start_time = time.time()

data_generator(df)
print(f"Tempo totale operazione: {time.time() - start_time:.2f} secondi")

Inizio caricamento dati con filtro in tempo reale...
Record totali da processare: 1,004,991
  100,000 record scritti...
  100,000 record scritti...
  200,000 record scritti...
  200,000 record scritti...
  300,000 record scritti...
  300,000 record scritti...
  400,000 record scritti...
  400,000 record scritti...
  500,000 record scritti...
  500,000 record scritti...
  600,000 record scritti...
  600,000 record scritti...
  700,000 record scritti...
  700,000 record scritti...
  800,000 record scritti...
  800,000 record scritti...
  900,000 record scritti...
  900,000 record scritti...
  1,000,000 record scritti...
  1,000,000 record scritti...

nCaricamento completato!
   Crimini scritti: 1,002,751
   Record scartati:
     - Coordinate nulle: 2,240
     - Data non valida: 0
     - Codice crimine mancante: 0
Tempo totale operazione: 116.70 secondi

nCaricamento completato!
   Crimini scritti: 1,002,751
   Record scartati:
     - Coordinate nulle: 2,240
     - Data non valida: 0
    

In [ ]:
# Verifica conteggio record in InfluxDB
q_count = f'''
from(bucket: "{bucket}")
  |> range(start: 2020-01-01T00:00:00Z, stop: 2026-01-01T00:00:00Z)
  |> filter(fn: (r) => r["_measurement"] == "crime_report")
  |> filter(fn: (r) => r["_field"] == "dr_no")
  |> count()
'''

result = query_api.query_data_frame(q_count)
if isinstance(result, list):
    result = pd.concat(result) if result else pd.DataFrame()

total_in_db = result['_value'].sum() if not result.empty else 0

print(f"Record attesi (da analisi CSV): 1,002,751")
print(f"Record effettivi in InfluxDB:   {total_in_db:,}")
print(f"Differenza:                     {1_002_751 - total_in_db:,}")

if total_in_db == 1_002_751:
    print("\nPERFETTO! Tutti i record sono stati caricati correttamente!")
elif total_in_db > 0:
    pct = total_in_db / 1_002_751 * 100
    print(f"\nCaricati {pct:.1f}% dei record attesi")

VERIFICA RECORD IN INFLUXDB
Record attesi (da analisi CSV): 1,002,751
Record effettivi in InfluxDB:   1,002,751
Differenza:                     0

✅ PERFETTO! Tutti i record sono stati caricati correttamente!


Alerting

In [ ]:
# =============================
# SISTEMA DI ALERTING (come da consegna)
# =============================
# "Impostare regole di warning e di allarme con soglie pari al 5% e 10%"

print("=" * 60)
print("SISTEMA DI ALERTING")
print("=" * 60)

# Calcolo soglie basate sui dati storici
daily_counts = df.groupby(df['DATE OCC'].dt.date).size()
mean_daily = daily_counts.mean()
max_daily = daily_counts.max()

# Soglie: percentili (più robusto del max)
SOGLIA_WARNING = daily_counts.quantile(0.95)  # 95° percentile
SOGLIA_ALARM = daily_counts.quantile(0.99)    # 99° percentile

print(f"\n📈 Statistiche giornaliere:")
print(f"   Media crimini/giorno: {mean_daily:.0f}")
print(f"   Max crimini/giorno: {max_daily}")
print(f"\n⚠️ Soglie Alert:")
print(f"   WARNING (95° percentile): >{SOGLIA_WARNING:.0f} crimini/giorno")
print(f"   ALARM (99° percentile): >{SOGLIA_ALARM:.0f} crimini/giorno")

# Identifica giorni con alert
warnings_days = daily_counts[(daily_counts > SOGLIA_WARNING) & (daily_counts <= SOGLIA_ALARM)]
alarm_days = daily_counts[daily_counts > SOGLIA_ALARM]

print(f"\n🔔 Alert rilevati:")
print(f"   Giorni WARNING: {len(warnings_days)}")
print(f"   Giorni ALARM: {len(alarm_days)}")

if len(alarm_days) > 0:
    print(f"\n🚨 Top 5 giorni ALARM:")
    for date, count in alarm_days.nlargest(5).items():
        print(f"   {date}: {count} crimini")

## 2. Interrogazioni Analitiche

Eseguiamo le analisi richieste utilizzando `query_api` e visualizzando i risultati con `matplotlib`.

In [ ]:
# =============================
# Q1: Numero medio giornaliero di crimini per Area e Stagione
# =============================

print("=" * 60)
print("Q1: Media giornaliera crimini per Area e Stagione")
print("=" * 60)

q1 = f'''
from(bucket: "{bucket}")
  |> range(start: 2020-01-01T00:00:00Z, stop: 2025-12-31T23:59:59Z)
  |> filter(fn: (r) => r["_measurement"] == "crime_report")
  |> filter(fn: (r) => r["_field"] == "dr_no")
  |> group(columns: ["area_name", "season"])
  |> count()
'''

start_time = time.time()
result_q1 = query_api.query_data_frame(q1)
query_time = time.time() - start_time
print(f"⏱️ Tempo query: {query_time:.2f}s")

if isinstance(result_q1, list):
    result_q1 = pd.concat(result_q1)

if not result_q1.empty:
    # Calcolo giorni per stagione (circa 90 giorni * ~5 anni = 450)
    # Ma calcoliamo più precisamente dal dataset
    df['Season'] = df['DATE OCC'].apply(get_season)
    days_per_season = df.groupby('Season')['DATE OCC'].apply(lambda x: x.dt.date.nunique())
    
    result_q1['daily_avg'] = result_q1.apply(
        lambda row: row['_value'] / days_per_season.get(row['season'], 450), axis=1
    )
    
    pivot_q1 = result_q1.pivot(index='area_name', columns='season', values='daily_avg')
    
    # Ordina colonne stagionali
    season_order = ['Winter', 'Spring', 'Summer', 'Autumn']
    pivot_q1 = pivot_q1[[s for s in season_order if s in pivot_q1.columns]]
    
    # Grafico
    fig, ax = plt.subplots(figsize=(14, 6))
    pivot_q1.plot(kind='bar', ax=ax, width=0.8, colormap='tab20')
    ax.set_title('Q1: Media Giornaliera Crimini per Area e Stagione', fontsize=14)
    ax.set_ylabel('Media Crimini / Giorno')
    ax.set_xlabel('Area LAPD')
    ax.legend(title='Stagione', bbox_to_anchor=(1.02, 1))
    ax.grid(axis='y', linestyle='--', alpha=0.5)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
    
    # Analisi picchi stagionali
    print("\n📊 Aree con variazione stagionale significativa (>15%):")
    for area in pivot_q1.index:
        max_val = pivot_q1.loc[area].max()
        min_val = pivot_q1.loc[area].min()
        variation = (max_val - min_val) / min_val * 100
        if variation > 15:
            max_season = pivot_q1.loc[area].idxmax()
            print(f"   {area}: picco in {max_season} (+{variation:.1f}%)")
else:
    print("⚠️ Nessun dato trovato")

In [ ]:
# =============================
# Q2: Distribuzione Part 1 vs Part 2 per Day/Night (Luglio 2023)
# =============================

print("=" * 60)
print("Q2: Distribuzione Day/Night per categoria (Luglio 2023)")
print("=" * 60)

q2 = f'''
from(bucket: "{bucket}")
  |> range(start: 2023-07-01T00:00:00Z, stop: 2023-07-31T23:59:59Z)
  |> filter(fn: (r) => r["_measurement"] == "crime_report")
  |> filter(fn: (r) => r["_field"] == "dr_no")
  |> group(columns: ["part", "time_period"])
  |> count()
'''

start_time = time.time()
result_q2 = query_api.query_data_frame(q2)
query_time = time.time() - start_time
print(f"⏱️ Tempo query: {query_time:.2f}s")

if isinstance(result_q2, list):
    result_q2 = pd.concat(result_q2)

if not result_q2.empty:
    # Mapping per leggibilità
    result_q2['Part_Label'] = result_q2['part'].map({'1': 'Part 1 (Gravi)', '2': 'Part 2 (Minori)'})
    result_q2['Time_Label'] = result_q2['time_period'].map({'Day': 'Giorno (6-18)', 'Night': 'Notte (18-6)'})
    
    pivot_q2 = result_q2.pivot(index='Part_Label', columns='Time_Label', values='_value')
    
    # Grafico
    fig, ax = plt.subplots(figsize=(10, 6))
    pivot_q2.plot(kind='bar', ax=ax, color=['#FFA500', '#4169E1'])
    ax.set_title('Q2: Crimini per Categoria e Fascia Oraria - Luglio 2023', fontsize=14)
    ax.set_ylabel('Numero Crimini')
    ax.set_xlabel('Categoria')
    ax.legend(title='Fascia Oraria')
    ax.grid(axis='y', linestyle='--', alpha=0.5)
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()
    
    # Analisi
    print("\n📊 Analisi:")
    for part in pivot_q2.index:
        night = pivot_q2.loc[part, 'Notte (18-6)'] if 'Notte (18-6)' in pivot_q2.columns else 0
        day = pivot_q2.loc[part, 'Giorno (6-18)'] if 'Giorno (6-18)' in pivot_q2.columns else 0
        total = night + day
        night_pct = night / total * 100 if total > 0 else 0
        print(f"   {part}: {night_pct:.1f}% notturni ({int(night):,} su {int(total):,})")
else:
    print("⚠️ Nessun dato trovato")

In [ ]:
# =============================
# Q3: Giornata con max crimini violenti per Area
# =============================

print("=" * 60)
print("Q3: Giornata con massimo crimini violenti (Part 1) per Area")
print("=" * 60)

q3 = f'''
from(bucket: "{bucket}")
  |> range(start: 2020-01-01T00:00:00Z, stop: 2025-12-31T23:59:59Z)
  |> filter(fn: (r) => r["_measurement"] == "crime_report")
  |> filter(fn: (r) => r["part"] == "1")
  |> filter(fn: (r) => r["_field"] == "dr_no")
  |> group(columns: ["area_name"])
  |> aggregateWindow(every: 1d, fn: count, createEmpty: false)
  |> max()
'''

try:
    start_time = time.time()
    result_q3 = query_api.query_data_frame(q3)
    query_time = time.time() - start_time
    print(f"⏱️ Tempo query: {query_time:.2f}s")
    
    if isinstance(result_q3, list):
        result_q3 = pd.concat(result_q3)
    
    if not result_q3.empty:
        # Prepara tabella
        display_q3 = result_q3[['area_name', '_time', '_value']].copy()
        display_q3['_time'] = pd.to_datetime(display_q3['_time']).dt.date
        display_q3.columns = ['Area', 'Data Picco', 'Crimini Violenti']
        display_q3 = display_q3.sort_values('Crimini Violenti', ascending=False)
        
        print("\n📋 Tabella: Giornata con Max Crimini Violenti per Area")
        print("=" * 55)
        print(f"{'Area':<20} {'Data':<15} {'Crimini':<10}")
        print("-" * 55)
        for _, row in display_q3.iterrows():
            print(f"{row['Area']:<20} {str(row['Data Picco']):<15} {int(row['Crimini Violenti']):<10}")
        print("=" * 55)
        
        # Analisi mesi
        display_q3['Mese'] = pd.to_datetime(display_q3['Data Picco']).dt.month_name()
        month_counts = display_q3['Mese'].value_counts()
        
        print("\n📊 Distribuzione picchi per mese:")
        for month, count in month_counts.head(5).items():
            print(f"   {month}: {count} aree")
    else:
        print("⚠️ Nessun dato trovato")
        
except Exception as e:
    print(f"❌ Errore: {e}")

In [ ]:
# =============================
# Q4: Età media vittime per Categoria e Anno
# =============================

print("=" * 60)
print("Q4: Età media vittime per categoria (Part) e anno")
print("=" * 60)

q4 = f'''
from(bucket: "{bucket}")
  |> range(start: 2020-01-01T00:00:00Z, stop: 2025-12-31T23:59:59Z)
  |> filter(fn: (r) => r["_measurement"] == "crime_report")
  |> filter(fn: (r) => r["_field"] == "vict_age")
  |> filter(fn: (r) => r["_value"] > 0 and r["_value"] < 120)
  |> group(columns: ["part"])
  |> aggregateWindow(every: 1y, fn: mean, createEmpty: false)
'''

try:
    start_time = time.time()
    result_q4 = query_api.query_data_frame(q4)
    query_time = time.time() - start_time
    print(f"⏱️ Tempo query: {query_time:.2f}s")
    
    if isinstance(result_q4, list):
        result_q4 = pd.concat(result_q4)
    
    if not result_q4.empty:
        result_q4['Year'] = pd.to_datetime(result_q4['_time']).dt.year
        result_q4['Part_Label'] = result_q4['part'].map({'1': 'Part 1 (Gravi)', '2': 'Part 2 (Minori)'})
        
        pivot_q4 = result_q4.pivot(index='Year', columns='Part_Label', values='_value')
        
        # Tabella
        print("\n📋 Età Media Vittime per Anno:")
        print(pivot_q4.round(1).to_string())
        
        # Grafico
        fig, ax = plt.subplots(figsize=(10, 6))
        for col in pivot_q4.columns:
            ax.plot(pivot_q4.index, pivot_q4[col], marker='o', linewidth=2, label=col)
        
        ax.set_title('Q4: Età Media Vittime per Categoria e Anno', fontsize=14)
        ax.set_xlabel('Anno')
        ax.set_ylabel('Età Media')
        ax.legend()
        ax.grid(True, linestyle='--', alpha=0.5)
        ax.set_xticks(pivot_q4.index)
        plt.tight_layout()
        plt.show()
    else:
        print("⚠️ Nessun dato trovato")
        
except Exception as e:
    print(f"❌ Errore: {e}")

### Performance Check [cite: 31, 32]
Confrontiamo il tempo di esecuzione di una query globale vs una filtrata per stagione.

In [ ]:
# =============================
# BENCHMARK: Query Globale vs Stagionale
# =============================

print("=" * 60)
print("BENCHMARK: Confronto tempi query globale vs stagionale")
print("=" * 60)

def measure_query(season_filter=None):
    """Misura tempo di esecuzione query con/senza filtro stagione."""
    base_q = f'''
    from(bucket: "{bucket}")
      |> range(start: 2020-01-01T00:00:00Z, stop: 2025-12-31T23:59:59Z)
      |> filter(fn: (r) => r["_measurement"] == "crime_report")
      |> filter(fn: (r) => r["_field"] == "dr_no")
    '''
    if season_filter:
        base_q += f'\n      |> filter(fn: (r) => r["season"] == "{season_filter}")'
    
    base_q += '\n      |> count()'
    
    start = time.time()
    query_api.query(base_q)
    return time.time() - start

# Esegui benchmark
print("\n🔍 Esecuzione benchmark...")

t_global = measure_query()
print(f"   Query globale: {t_global:.3f}s")

seasons = ['Winter', 'Spring', 'Summer', 'Autumn']
seasonal_times = {}
for season in seasons:
    t = measure_query(season)
    seasonal_times[season] = t
    print(f"   Query {season}: {t:.3f}s")

avg_seasonal = np.mean(list(seasonal_times.values()))

print(f"\n📊 Risultati:")
print(f"   Tempo globale: {t_global:.3f}s")
print(f"   Tempo medio stagionale: {avg_seasonal:.3f}s")

if t_global > avg_seasonal:
    speedup = (t_global - avg_seasonal) / t_global * 100
    print(f"   ✅ Speedup filtro Season: {speedup:.1f}%")
else:
    print(f"   ⚠️ Nessun speedup significativo (dati troppo piccoli o cache)")

# Grafico
fig, ax = plt.subplots(figsize=(10, 5))
times = [t_global] + list(seasonal_times.values())
labels = ['Globale'] + seasons
colors = ['#2E86AB'] + ['#A23B72']*4

ax.bar(labels, times, color=colors)
ax.set_title('Benchmark: Tempo Query Globale vs Stagionale', fontsize=14)
ax.set_ylabel('Tempo (secondi)')
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

--- 
## 5. Analisi Avanzata: Clustering Incrementale

Utilizziamo `MiniBatchKMeans` per raggruppare le aree in base ai pattern di criminalità aggregati (Area, Stagione, Anno).
Features: Totale Crimini, % Arma, % Notte, % Violenti[cite: 33].

In [ ]:
# =============================
# CLUSTERING INCREMENTALE
# =============================
# Feature per ogni (Area, Season, Year):
# - Totale crimini
# - % crimini con arma
# - % crimini notturni
# - % crimini violenti (Part 1)

print("=" * 60)
print("CLUSTERING: Estrazione feature da InfluxDB")
print("=" * 60)

# Query totale crimini per Area/Season/Year
q_total = f'''
from(bucket: "{bucket}")
  |> range(start: 2020-01-01T00:00:00Z, stop: 2025-12-31T23:59:59Z)
  |> filter(fn: (r) => r["_measurement"] == "crime_report")
  |> filter(fn: (r) => r["_field"] == "dr_no")
  |> group(columns: ["area_name", "season"])
  |> aggregateWindow(every: 1y, fn: count, createEmpty: false)
'''

# Query crimini con arma
q_weapon = f'''
from(bucket: "{bucket}")
  |> range(start: 2020-01-01T00:00:00Z, stop: 2025-12-31T23:59:59Z)
  |> filter(fn: (r) => r["_measurement"] == "crime_report")
  |> filter(fn: (r) => r["_field"] == "dr_no")
  |> filter(fn: (r) => r["weapon_flag"] == "True")
  |> group(columns: ["area_name", "season"])
  |> aggregateWindow(every: 1y, fn: count, createEmpty: false)
'''

# Query crimini notturni
q_night = f'''
from(bucket: "{bucket}")
  |> range(start: 2020-01-01T00:00:00Z, stop: 2025-12-31T23:59:59Z)
  |> filter(fn: (r) => r["_measurement"] == "crime_report")
  |> filter(fn: (r) => r["_field"] == "dr_no")
  |> filter(fn: (r) => r["time_period"] == "Night")
  |> group(columns: ["area_name", "season"])
  |> aggregateWindow(every: 1y, fn: count, createEmpty: false)
'''

# Query crimini violenti (Part 1)
q_violent = f'''
from(bucket: "{bucket}")
  |> range(start: 2020-01-01T00:00:00Z, stop: 2025-12-31T23:59:59Z)
  |> filter(fn: (r) => r["_measurement"] == "crime_report")
  |> filter(fn: (r) => r["_field"] == "dr_no")
  |> filter(fn: (r) => r["part"] == "1")
  |> group(columns: ["area_name", "season"])
  |> aggregateWindow(every: 1y, fn: count, createEmpty: false)
'''

print("🔍 Esecuzione query per feature clustering...")
start_time = time.time()

df_total = query_api.query_data_frame(q_total)
df_weapon = query_api.query_data_frame(q_weapon)
df_night = query_api.query_data_frame(q_night)
df_violent = query_api.query_data_frame(q_violent)

print(f"⏱️ Query completate in {time.time() - start_time:.2f}s")

# Concatena se lista
for name, df_q in [('df_total', df_total), ('df_weapon', df_weapon), 
                    ('df_night', df_night), ('df_violent', df_violent)]:
    if isinstance(df_q, list):
        exec(f"{name} = pd.concat(df_q)")

# Prepara DataFrame
def prepare_df(df_q, col_name):
    df_q = df_q.copy()
    df_q['Year'] = pd.to_datetime(df_q['_time']).dt.year
    df_q = df_q.rename(columns={'_value': col_name})
    return df_q[['area_name', 'season', 'Year', col_name]]

df_total = prepare_df(df_total, 'total')
df_weapon = prepare_df(df_weapon, 'weapon')
df_night = prepare_df(df_night, 'night')
df_violent = prepare_df(df_violent, 'violent')

# Merge
df_features = df_total.merge(df_weapon, on=['area_name', 'season', 'Year'], how='left')
df_features = df_features.merge(df_night, on=['area_name', 'season', 'Year'], how='left')
df_features = df_features.merge(df_violent, on=['area_name', 'season', 'Year'], how='left')
df_features = df_features.fillna(0)

# Calcola percentuali
df_features['pct_weapon'] = df_features['weapon'] / df_features['total'] * 100
df_features['pct_night'] = df_features['night'] / df_features['total'] * 100
df_features['pct_violent'] = df_features['violent'] / df_features['total'] * 100

print(f"\n📊 Feature estratte: {len(df_features)} record (Area x Season x Year)")
print(df_features.head(10))

In [ ]:
# =============================
# CLUSTERING CON MiniBatchKMeans
# =============================

print("=" * 60)
print("CLUSTERING INCREMENTALE con MiniBatchKMeans")
print("=" * 60)

# Feature per clustering
feature_cols = ['total', 'pct_weapon', 'pct_night', 'pct_violent']
n_clusters = 4

# Normalizzazione
scaler = StandardScaler()

# Separa train (2020-2024) e test (2025)
train_mask = df_features['Year'] < 2025
df_train = df_features[train_mask].copy()
df_test = df_features[~train_mask].copy()

print(f"   Train set: {len(df_train)} record (2020-2024)")
print(f"   Test set: {len(df_test)} record (2025)")

# Fit scaler e clustering su train
X_train = scaler.fit_transform(df_train[feature_cols])

kmeans = MiniBatchKMeans(n_clusters=n_clusters, random_state=42, batch_size=256)
kmeans.fit(X_train)
df_train['Cluster'] = kmeans.labels_

# Predict su test
if len(df_test) > 0:
    X_test = scaler.transform(df_test[feature_cols])
    df_test['Cluster'] = kmeans.predict(X_test)

# Unisci
df_clustered = pd.concat([df_train, df_test])

# === ANALISI CENTROIDI ===
print("\n📊 Centroidi dei Cluster (valori originali):")
centroids = pd.DataFrame(
    scaler.inverse_transform(kmeans.cluster_centers_), 
    columns=feature_cols
)
centroids.index.name = 'Cluster'
print(centroids.round(2).to_string())

# Interpretazione cluster
print("\n💡 Interpretazione Cluster:")
for c_id in range(n_clusters):
    row = centroids.loc[c_id]
    crime_level = "ALTO" if row['total'] > centroids['total'].median() else "BASSO"
    violent_level = "ALTO" if row['pct_violent'] > centroids['pct_violent'].median() else "BASSO"
    weapon_level = "ALTO" if row['pct_weapon'] > centroids['pct_weapon'].median() else "BASSO"
    print(f"   Cluster {c_id}: Volume {crime_level}, Violenza {violent_level}, Armi {weapon_level}")

# === CONFRONTO 2020 vs 2024 ===
print("\n" + "=" * 60)
print("CONFRONTO CLUSTER: 2020 vs 2024")
print("=" * 60)

cluster_dist = df_clustered.groupby(['Year', 'Cluster']).size().unstack(fill_value=0)
print(cluster_dist)

# Grafico distribuzione per anno
fig, ax = plt.subplots(figsize=(12, 6))
cluster_dist.plot(kind='bar', stacked=True, ax=ax, colormap='viridis')
ax.set_title('Distribuzione Aree nei Cluster per Anno', fontsize=14)
ax.set_ylabel('Numero (Area x Season)')
ax.set_xlabel('Anno')
ax.legend(title='Cluster', bbox_to_anchor=(1.02, 1))
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# === SCATTER PLOT 2020 vs 2024 ===
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
colors = ['#E74C3C', '#3498DB', '#2ECC71', '#9B59B6']

for idx, year in enumerate([2020, 2024]):
    ax = axes[idx]
    df_year = df_clustered[df_clustered['Year'] == year]
    
    for c_id in range(n_clusters):
        data = df_year[df_year['Cluster'] == c_id]
        ax.scatter(data['pct_weapon'], data['pct_violent'], 
                   c=colors[c_id], label=f'Cluster {c_id}', s=80, alpha=0.7)
    
    ax.set_xlabel('% Crimini con Arma')
    ax.set_ylabel('% Crimini Violenti')
    ax.set_title(f'Clustering Aree - {year}', fontsize=14)
    ax.legend()
    ax.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# =============================
# CHIUSURA E RIEPILOGO
# =============================

client.close()

print("=" * 70)
print("📊 RIEPILOGO ANALISI CRIMINI LOS ANGELES (2020-2025)")
print("=" * 70)

print("""
✅ CARICAMENTO DATI:
   - Schema: 7 TAG (area_name, area_id, part, weapon_flag, season, time_period, vict_sex)
   - Timestamp: US con DR_NO per unicità (99.9% crimini salvati)
   - Cardinalità TAG totale: ~672 combinazioni

✅ CONTEGGIO SIMULATO:
   - Per Area, Classificazione, Con/Senza Arma
   - (Distretto disponibile come FIELD per query specifiche)

✅ SISTEMA ALERTING:
   - Soglie basate su percentili (95° warning, 99° alarm)

✅ QUERY ANALITICHE:
   Q1: Media giornaliera crimini per Area/Stagione
   Q2: Distribuzione Day/Night per categoria (Luglio 2023)
   Q3: Giornata picco crimini violenti per Area
   Q4: Età media vittime per categoria/anno

✅ BENCHMARK:
   - Confronto query globale vs filtrata per Season

✅ CLUSTERING INCREMENTALE:
   - MiniBatchKMeans (4 cluster)
   - Feature: totale, % arma, % notte, % violenti
   - Confronto evoluzione 2020 vs 2024
""")

print("=" * 70)
print("✅ Notebook completato - Connessione chiusa")